# Bag of Words Exploration Notebook

## 1. Setup 

In [ ]:
import os
import glob
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from convert_to_txt import convert_word_to_text, extract_excel_values
from openpyxl import load_workbook
from openpyxl.utils.cell import coordinate_from_string, column_index_from_string
import pypandoc
import matplotlib.pyplot as plt

In [ ]:
# Directory containing .txt files
TEXT_DIR = "text" 

# Directory containing other metadata
DATA_DIR = "data"

### Convert word docs or excel files to text files

In [ ]:
# convert_word_to_text(
#     word_file="input.docx",
#     txt_file="output.txt"
# )

In [ ]:
extract_excel_values(
    excel_file=f"{TEXT_DIR}/hw7_solutions.xlsx",
    txt_file=f"{TEXT_DIR}/human_responses.txt",
    start_cell="G2",
    end_cell="G522",
    increment=5
)

In [ ]:
extract_excel_values(
    excel_file=f"{TEXT_DIR}/hw7_solutions.xlsx",
    txt_file=f"{TEXT_DIR}/AI_responses.txt",
    start_cell="G3",
    end_cell="G523",
    increment=5
)

## 2. Load documents 

In [ ]:
filepaths = glob.glob(os.path.join(TEXT_DIR, "*.txt"))

corpus = []
filenames = []

for fp in filepaths:
    with open(fp, 'r', encoding='utf-8', errors='ignore') as f:
        corpus.append(f.read())
        filenames.append(os.path.basename(fp))

print(f"Loaded {len(corpus)} documents")

## 3. Create Bag-of-Words representation 

In [ ]:
# https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html

vectorizer = CountVectorizer(
    strip_accents='unicode',
    stop_words='english',
    max_df=2,
    min_df=1
)

X = vectorizer.fit_transform(corpus)

print("Matrix shape:", X.shape)

## 4. Convert to DataFrame for exploration 

In [ ]:
feature_names = vectorizer.get_feature_names_out()

bow_df = pd.DataFrame(X.toarray(), columns=feature_names, index=filenames)

bow_df.head()

## 5. Most common words overall 

In [ ]:
word_counts = bow_df.sum(axis=0).sort_values(ascending=False)

print("Top 20 words:")
print(word_counts.head(20))

## 6. Most important words per document 

In [ ]:
def top_words_per_doc(df, n=20):
    results = {}
    for doc in df.index:
        top_words = df.loc[doc].sort_values(ascending=False).head(n)
        results[doc] = top_words[top_words > 0]
    return results

per_doc = top_words_per_doc(bow_df)

In [ ]:
# Example: show one document
example_doc = list(per_doc.keys())[0]
print(f"\nTop words in {example_doc}:")
print(per_doc[example_doc])

In [ ]:
# Example: show one document
example_doc = list(per_doc.keys())[1]
print(f"\nTop words in {example_doc}:")
print(per_doc[example_doc])

## 7. Visualize top words 

In [ ]:
word_counts.head(20).plot(kind='bar')
plt.title("Top 20 Words Across Corpus")
plt.xlabel("Words")
plt.ylabel("Frequency")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Optional: Save outputs 

In [ ]:
# bow_df.to_csv("outputs/bow_matrix.csv")
# sim_df.to_csv("outputs/document_similarity.csv")

# print("Saved outputs: bow_matrix.csv, document_similarity.csv")